# NB0 - Build Your First Agent (and Meet the Harness)

**Workshop: Self-Evolving Agents by Optimizing the Harness (no GPU)**

Before we can make an agent *improve itself*, we have to build one. By the end of
this notebook you'll have a working **text-to-SQL agent** - and you'll have built
it piece by piece, so the rest of the day has a name for every part we evolve.

The one idea to hold onto: **an agent = a brain + a harness.**
- **Brain** = the LLM weights. We *never* touch them - fine-tuning the brain is
  what needs a GPU, and the whole point of this workshop is to avoid that.
- **Harness** = everything *around* the brain: the prompt/context, the tools, the
  memory, the control loop. This is the part we build now and optimize all day.

We'll assemble the harness in four moves and name each one using the framework
**H = (E, T, C, S, L, V)** = Execution loop, Tool registry, Context manager,
State store, Lifecycle hooks, e**V**aluation interface.

In [ ]:
# Setup. We run from the notebooks/ folder, so add the repo root to the path.
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from workshop_utils import (
    build_db, run_sql, llm, METER, SCHEMA_TEXT, extract_sql, baseline_prompt,
    preflight, flush,
)

preflight()              # hard-require OPENAI + LANGFUSE keys (see SETUP.md)
DB = build_db()          # deterministic rebuild; same data on every machine
print("Database ready at:", DB)

> **Observability is on.** From here, *every* `llm()` call is traced to
> **Langfuse** - open your project at the `LANGFUSE_BASE_URL` you configured and
> you'll see each call appear live. That dashboard is how a real team inspects an
> agent's behaviour, cost, and latency; we'll lean on it throughout.

## Meet the database first

Before we ask the model for any SQL, let's *look* at the database it will be
querying - so the table and column names mean something to you, and you can spot
when the model gets them wrong. It's a small toy **"shop"**: customers place
orders, each order has line items, and each line item points at a product.

In [ ]:
# A quick tour of the four tables: the schema, then row counts + a few sample rows.
print(SCHEMA_TEXT)

def peek(table, n=4):
    rows, _ = run_sql(f"SELECT * FROM {table} LIMIT {n}")
    total, _ = run_sql(f"SELECT COUNT(*) FROM {table}")
    print(f"\n{table}  -  {total[0][0]} rows total (showing {len(rows)}):")
    for r in rows:
        print("   ", r)

for t in ("customers", "products", "orders", "order_items"):
    peek(t)

## Move 1 - The brain, alone

The simplest thing we can do is ask the LLM for SQL with **no schema and no
tools**. Watch what happens: it has to *guess* our table and column names. It
might look confident and still be completely wrong - and nothing here can tell.

In [ ]:
question = "How many customers are in Mumbai?"
raw = llm(f"Write a SQLite query for this request: {question}")
print(raw)
# This is not an agent. It is a single guess, ungrounded in our actual database.

## Move 2 - Give the brain context (C)

The model can't know *our* schema unless we put it in the prompt. The
**context manager (C)** is the part of the harness that decides what the brain
sees. `baseline_prompt` just formats `schema + question` into messages.

In [ ]:
msgs = baseline_prompt(question)
print(msgs[1]["content"])          # the exact context we send the brain
print("\n--- model's SQL ---")
sql = extract_sql(llm(msgs))       # extract_sql pulls the query out of the reply
print(sql)
# Better - now it uses real columns. But it's still just text in, text out:
# if this query is wrong, the agent has no way to find out.

## Move 3 - Give it a tool, let it act (T + E)

An agent doesn't just *emit text* - it **acts on the world and observes the
result**. Our world is the database; our **tool (T)** is `run_sql`, which
executes a query and returns either rows or an error string. Calling that tool is
the first turn of the **execution loop (E)**.

In [ ]:
rows, err = run_sql(sql)
print("error :", err)
print("rows  :", rows)

# Now deliberately break a query to see the feedback the agent will react to:
bad_rows, bad_err = run_sql("SELECT name FROM custmers WHERE city='Mumbai'")
print("\nbroken query -> error:", bad_err)
# The environment talks back. That error is a free learning signal - no labels.

## Move 4 - Close the loop (E + L)

Here's the agentic part: if the tool reports an error, feed that error back to
the brain and let it try again. **generate -> execute -> observe -> retry.** That
feedback loop is what turns a single call into an *agent*. (`L`, lifecycle hooks,
is just *when* we stop: here, on success or after N tries.)

In [ ]:
REPAIR_SYS = ("You are a SQLite expert. A query failed; return a corrected "
              "query inside a ```sql code block.")

def my_first_agent(question, max_repairs=2):
    sql = extract_sql(llm(baseline_prompt(question)))
    for attempt in range(max_repairs + 1):
        rows, err = run_sql(sql)
        if err is None:
            print(f"  [solved on attempt {attempt}]")
            return sql
        print(f"  [attempt {attempt} failed: {err}] -> reflecting and retrying")
        fix = llm([
            {"role": "system", "content": REPAIR_SYS},
            {"role": "user", "content":
                f"Schema:\n{SCHEMA_TEXT}\nQuestion: {question}\n\n"
                f"Failed SQL:\n{sql}\n\nDatabase error: {err}"},
        ])
        sql = extract_sql(fix)
    return sql

print("FINAL:", my_first_agent("How many customers are in Mumbai?"))

## What you just built

You built an agent. Map it back to the framework:

| Component | What we built |
|---|---|
| **C** - context manager | the schema, injected into the prompt |
| **T** - tool registry | `run_sql` (execute a query against the DB) |
| **E** - execution loop | generate -> run -> observe -> retry |
| **L** - lifecycle hooks | stop on success, or after N tries |
| **S** - state store | *(empty)* - the agent forgets everything between questions |
| **V** - evaluation interface | *(none yet)* - we can't measure how good it is |

`S` and `V` are blank. That is **not** an accident - filling them in is the whole
workshop. `V` (a reward signal) comes next in NB1; `S` (memory and skills that
persist and compound) is NB2 onward.

In [ ]:
# We've packaged exactly this agent as `make_agent`, so every later notebook
# reuses the same harness. The `extra=` slot is where memory / skills (S) plug in.
from workshop_utils import make_agent

agent = make_agent()
print(agent("List the names of customers in the Enterprise segment."))
print("\n", METER)
flush()                  # send the buffered traces to Langfuse
# Open Langfuse now: you'll see a `sql_agent` trace per question, with the
# initial call and any repair calls nested inside it - that's the trajectory.

## Takeaways

- An **agent = brain + harness**. We froze the brain and built a harness around
  it: context (**C**), a tool (**T**), and an execution loop (**E**).
- The loop already handles *crashes* for free, with no labels - that's real, but
  it's the easy part.
- Two things are still missing: we can't **measure** quality (**V**), and the
  agent has no **memory** (**S**) - every question starts from zero.

**Next - NB1:** build the evaluation interface **V** and measure the agent you
just built. *You can't improve what you can't measure.*